# Sliding Window Attention — Solution

Reference implementation for `01_sliding_window_exercise.ipynb`.

## Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Solution 1 — `sliding_window_mask`

In [2]:
def sliding_window_mask(seq_len, window_size):
    mask = torch.arange(seq_len)[:,None] - torch.arange(seq_len)[None,:]
    mask = torch.where((mask>=window_size) | (mask<0),False,True)
    
    return mask

In [3]:
sliding_window_mask(6, 3)

tensor([[ True, False, False, False, False, False],
        [ True,  True, False, False, False, False],
        [ True,  True,  True, False, False, False],
        [False,  True,  True,  True, False, False],
        [False, False,  True,  True,  True, False],
        [False, False, False,  True,  True,  True]])

## Solution 2 — `SlidingWindowAttention`

In [4]:
class SlidingWindowAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_len, window_size):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.window_size = window_size

        self.d_head = d_model//n_heads 

        self.qkv_proj = nn.Linear(d_model,d_model*3)
        self.out_proj = nn.Linear(d_model,d_model)

        mask = torch.arange(max_len)[:,None] - torch.arange(max_len)[None,:]
        self.register_buffer("sliding_window_mask",torch.where((mask>=window_size) | (mask<0),True,False))

    def forward(self, x):
        bs,seq_len,_ = x.shape

        Q,K,V = torch.chunk(self.qkv_proj(x),3,dim=-1) #bs,seq_len,d_model
        
        Q = Q.view(bs,seq_len,self.n_heads,self.d_head).permute(0,2,1,3) #bs,n_heads,seq_len,d_head
        K = K.view(bs,seq_len,self.n_heads,self.d_head).permute(0,2,1,3)
        V = V.view(bs,seq_len,self.n_heads,self.d_head).permute(0,2,1,3)
        
        scores = Q@K.transpose(-2,-1) / self.d_head**0.5

        ## Sliding Mask and causal Mask fill
        scores = scores.masked_fill(self.sliding_window_mask[:seq_len,:seq_len],float("-inf"))

        weights = torch.softmax(scores,dim=-1)

        output = weights@V # bs,n_heads,seq_len,d_head
        output = output.permute(0,2,1,3).contiguous().view(bs,seq_len,self.d_model)

        output = self.out_proj(output)

        return output

**Why does this still work?** Receptive field grows with depth. After L layers each with window w, a token at position N can transitively reach tokens at distance L*w away — the bucket-brigade effect. Information passes through intermediate tokens as relays.

## Solution 3 — Locality verification

In [12]:
x_a = torch.randn((1, 8, 64))
x_b = x_a.clone()
x_b[:,1,:] = torch.randn(64)

attention = SlidingWindowAttention( 
    d_model = 64, 
    n_heads = 2, 
    max_len = 8,
    window_size=3
)
out_a = attention(x_a)
out_b = attention(x_b)

assert torch.allclose(out_a[0, 5], out_b[0, 5])
assert torch.allclose(out_a[0, 4], out_b[0, 4])
assert not torch.allclose(out_a[0, 3], out_b[0, 3])

## Run the tests

In [6]:
from tests import run_sliding_window_tests
run_sliding_window_tests(sliding_window_mask, SlidingWindowAttention)

Running SlidingWindowAttention...
  ✓ Mask shape correct
  ✓ Mask diagonal is True
  ✓ Mask respects window size
  ✓ Mask is causal (no future)
  ✓ Module output shape correct
  ✓ Locality: distant tokens don't leak

  All 6 checks passed ✓



True